# Crazyflie hover with Brax PPO
Install the cuda13 and train extras and select the repository uv Python environment as your notebook kernel. The policy learns CTBR actions through motor dynamics, targeting (0, 0, 0.6) m. Two million steps is a starting budget, not a convergence guarantee.

In [ ]:
import os
os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
# Set CUDA_VISIBLE_DEVICES before importing JAX if needed.
from pathlib import Path
import sys
example_dir = Path.cwd() if (Path.cwd() / 'train_hover_ppo.py').exists() else Path.cwd() / 'examples'
sys.path.insert(0, str(example_dir.resolve()))
from train_hover_ppo import train, load_policy, render_rollout
import jax
print(jax.default_backend(), jax.devices())
output = example_dir.parent / 'artifacts' / 'hover_ppo'

In [ ]:
make_inference_fn, params, history = train(output, num_timesteps=2_000_000, num_envs=128, seed=32)

In [ ]:
import matplotlib.pyplot as plt
evaluations = [r for r in history if 'eval/episode_reward' in r]
plt.plot([r['steps'] for r in evaluations], [r['eval/episode_reward'] for r in evaluations])
plt.xlabel('Environment steps')
plt.ylabel('Evaluation episode return')
plt.grid(alpha=0.3)
plt.show()

## Reload and render
The following cells can also run after setup without training again. Load only trusted parameter files.

In [ ]:
make_inference_fn, params = load_policy(output / 'params.pkl')
video_path = render_rollout(make_inference_fn, params, output / 'rollout.mp4', steps=1000)
from IPython.display import Video, display
display(Video(str(video_path), embed=True))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
positions = np.load(video_path.with_suffix('.npz'))['positions']
time = (np.arange(len(positions)) + 1) * 0.002
plt.plot(time, positions, label=['x', 'y', 'z'])
plt.axhline(0.6, linestyle='--', color='gray', label='target z')
plt.xlabel('Time (s)')
plt.ylabel('Position (m)')
plt.legend()
plt.show()